<div align="center">
  
  <img src="https://github.com/JHermosillaD/Logos/blob/main/uv.png?raw=true" height="60">&nbsp;&nbsp;&nbsp;
  <img src="https://github.com/JHermosillaD/Logos/blob/main/lab_uv.png?raw=true" height="55">&nbsp;&nbsp;&nbsp;
  <img src="https://github.com/JHermosillaD/Logos/blob/main/python.png?raw=true" height="55">&nbsp;&nbsp;&nbsp;
  <img src="https://github.com/JHermosillaD/Logos/blob/main/opencv.png?raw=true" height="55">&nbsp;&nbsp;&nbsp;
  <img src="https://github.com/JHermosillaD/Logos/blob/main/mediapipe.png?raw=true" height="50">
  
  ---
  <h4>:D</h4>
  <h2>Computer Vision with MediaPipe</h2>
  <p><b>By Jesus E. Hermosilla-Diaz</b></p>
  
  <img src="https://private-user-images.githubusercontent.com/42659983/587271200-0acb06e0-46e7-452b-bb24-9ad81d17ea59.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3Nzc5Mjk2MzgsIm5iZiI6MTc3NzkyOTMzOCwicGF0aCI6Ii80MjY1OTk4My81ODcyNzEyMDAtMGFjYjA2ZTAtNDZlNy00NTJiLWJiMjQtOWFkODFkMTdlYTU5LnBuZz9YLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFLSUFWQ09EWUxTQTUzUFFLNFpBJTJGMjAyNjA1MDQlMkZ1cy1lYXN0LTElMkZzMyUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNTA0VDIxMTUzOFomWC1BbXotRXhwaXJlcz0zMDAmWC1BbXotU2lnbmF0dXJlPTg1YWI3OGRlMjQ0MDlkYTI4MmE1NDI4ZjIwMWY0ODg2YTlhYmQ5ZjIwNWQ5ZTE3MDBiMzkyYzUyODg3Yzk5OGQmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JnJlc3BvbnNlLWNvbnRlbnQtdHlwZT1pbWFnZSUyRnBuZyJ9.Ub4tB-qFBhmY5S50d__Wro6PpZUIMBKb_fxmyhwhegY" height="200">
  
  <hr>

<p><b>Resources:</b></p>


[MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker)


[Face Landmark Index - Full Size](https://storage.googleapis.com/mediapipe-assets/documentation/mediapipe_face_landmark_fullsize.png)

[Face Landmark Model Card](https://storage.googleapis.com/mediapipe-assets/Model%20Card%20MediaPipe%20Face%20Mesh%20V2.pdf)

[Blend Shape Model Card](https://storage.googleapis.com/mediapipe-assets/Model%20Card%20Blendshape%20V2.pdf)

[Markers](http://acodigo.blogspot.com/2017/06/funciones-de-dibujo-opencv-python.html)

</div>

In [ ]:
%%capture
!pip install -q mediapipe
!wget -O face_landmarker.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
# MediaPipe running engine
# PLEASE DO NOT TOUCH

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import cv2
from base64 import b64decode, b64encode
from google.colab import output
from IPython.display import display, Javascript

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=2,
    min_face_detection_confidence=0.5
)
detector = vision.FaceLandmarker.create_from_options(options)

def run_engine(filter_function):
    def process_and_return(data_url, *args, **kwargs):
        header, encoded = data_url.split(",", 1)
        img_bytes = b64decode(encoded)
        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        bgr = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if bgr is None:
            return ""

        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)

        annotated = np.copy(rgb)

        annotated = filter_function(annotated, result.face_landmarks)

        _, buf = cv2.imencode(".jpg", cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
        b64 = b64encode(buf).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"

    output.register_callback("notebook.process_frame", process_and_return)

    display(Javascript("""
    (async () => {
      const container = document.createElement('div');
      container.style.cssText = 'display:flex; flex-direction:column; align-items:center; gap:8px;';
      const label = document.createElement('div');
      label.textContent = 'Starting camera...';
      label.style.cssText = 'font-family:monospace; font-size:13px; color:#aaa;';
      const img = document.createElement('img');
      img.style.cssText = 'border-radius:8px; max-width:640px; width:100%;';
      const stopBtn = document.createElement('button');
      stopBtn.textContent = 'Stop';
      stopBtn.style.cssText = 'padding:6px 18px; font-size:14px; cursor:pointer; border-radius:6px;';
      container.append(label, img, stopBtn);
      document.body.appendChild(container);
      const video = document.createElement('video');
      video.style.display = 'none';
      document.body.appendChild(video);
      const stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      await video.play();
      await new Promise(r => setTimeout(r, 500));
      const canvas = document.createElement('canvas');
      canvas.width  = video.videoWidth  || 640;
      canvas.height = video.videoHeight || 480;
      const ctx = canvas.getContext('2d');
      label.textContent = 'Face mesh active';
      let running = true;
      stopBtn.onclick = () => { running = false; };
      while (running) {
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        const raw = canvas.toDataURL('image/jpeg', 0.8);
        const result = await google.colab.kernel.invokeFunction(
          'notebook.process_frame', [raw], {});
        const annotated = result.data['text/plain'].replace(/^'|'$/g, '');
        if (annotated.startsWith('data:image')) {
          img.src = annotated;
        }
      }
      stream.getVideoTracks()[0].stop();
      video.remove();
      label.textContent = 'Stopped';
      stopBtn.remove();
    })();
    """))

In [ ]:
import cv2
import numpy as np

# Helpers
def get_pixel_coords(landmark, img_w, img_h):
    return (int(landmark.x * img_w), int(landmark.y * img_h))
# -----------------------------------------------------------

# Example 1
def filter_example1(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()

    for face_landmarks in face_landmarks_list:
        for landmark in face_landmarks:
            pt = get_pixel_coords(landmark, w, h)
            cv2.circle(result, pt, 1, (0, 255, 0), -1)

    return result
# -----------------------------------------------------------

# Example 2
def filter_example2(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()

    for face_landmarks in face_landmarks_list:
        # Index 1 is the nose tip
        nose = get_pixel_coords(face_landmarks[1], w, h)
        cv2.circle(result, nose, 25, (255, 0, 0), -1)
        #cv2.drawMarker(result, nose, (255, 0, 0), markerType=cv2.MARKER_STAR,markerSize=50, thickness=2)

        # Index 159 is left eye top, 386 is right eye top)
        left_eye = get_pixel_coords(face_landmarks[159], w, h)
        right_eye = get_pixel_coords(face_landmarks[386], w, h)

        cv2.circle(result, left_eye, 20, (255, 255, 255), -1)
        cv2.circle(result, left_eye, 10, (0, 0, 0), -1)

        cv2.circle(result, right_eye, 15, (255, 255, 255), -1)
        cv2.circle(result, right_eye, 10, (0, 0, 0), -1)

        # Left corner (61), Bottom lip (17), Right corner (291)
        mouth_left = get_pixel_coords(face_landmarks[61], w, h)
        mouth_bottom = get_pixel_coords(face_landmarks[17], w, h)
        mouth_right = get_pixel_coords(face_landmarks[291], w, h)

        smile_pts = np.array([mouth_left, mouth_bottom, mouth_right], np.int32)
        cv2.polylines(result, [smile_pts], isClosed=False, color=(0, 0, 255), thickness=5)

    return result
# -----------------------------------------------------------

# Example 3
def filter_example3(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()

    for face_landmarks in face_landmarks_list:
        test1 = get_pixel_coords(face_landmarks[1], w, h)
        test2 = get_pixel_coords(face_landmarks[2], w, h)
        test3 = get_pixel_coords(face_landmarks[3], w, h)
        test4 = get_pixel_coords(face_landmarks[4], w, h)
        test_list = [test1, test2, test3, test4]
        pts = np.array(
            test_list,
            dtype=np.int32
        )
        hull = cv2.convexHull(pts)

        # Mask for hull region only
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillConvexPoly(mask, hull, 255)

        # Blur region
        blurred = cv2.GaussianBlur(result, (55, 55), 30)
        result = np.where(mask[:, :, np.newaxis] == 255, blurred, result)

    return result

# -----------------------------------------------------------

# Example 4
def filter_example4(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()

    # Left eye corners: 33 (outer), 133 (inner)
    # Right eye corners: 362 (inner), 263 (outer)
    for face_landmarks in face_landmarks_list:
        l_outer = get_pixel_coords(face_landmarks[33],  w, h)
        l_inner = get_pixel_coords(face_landmarks[133], w, h)
        r_inner = get_pixel_coords(face_landmarks[362], w, h)
        r_outer = get_pixel_coords(face_landmarks[263], w, h)

        eye_h = int(abs(
            get_pixel_coords(face_landmarks[159], w, h)[1] -
            get_pixel_coords(face_landmarks[145], w, h)[1]
        ) * 2.5) + 10
        pad = 8

        for outer, inner in [(l_outer, l_inner), (r_inner, r_outer)]:
            x1 = min(outer[0], inner[0]) - pad
            x2 = max(outer[0], inner[0]) + pad
            cy = (outer[1] + inner[1]) // 2
            cv2.rectangle(result, (x1, cy - eye_h//2), (x2, cy + eye_h//2), (10, 10, 10), -1)
            cv2.rectangle(result, (x1, cy - eye_h//2), (x2, cy + eye_h//2), (80, 60, 20), 2)

        # Bridges
        bridge_l = ((l_outer[0]+l_inner[0])//2, (l_outer[1]+l_inner[1])//2)
        bridge_r = ((r_inner[0]+r_outer[0])//2, (r_inner[1]+r_outer[1])//2)
        cv2.line(result, bridge_l, bridge_r, (80, 60, 20), 2)

    return result
# -----------------------------------------------------------

# Example 5
def filter_example5(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()

    for face_landmarks in face_landmarks_list:
        top_lip    = get_pixel_coords(face_landmarks[13],  w, h)
        bottom_lip = get_pixel_coords(face_landmarks[14],  w, h)
        l_eye      = get_pixel_coords(face_landmarks[33],  w, h)
        r_eye      = get_pixel_coords(face_landmarks[263], w, h)

        mouth_open  = abs(bottom_lip[1] - top_lip[1])
        eye_dist    = abs(r_eye[0] - l_eye[0])
        ratio       = mouth_open / (eye_dist + 1e-5)

        if ratio > 0.10:
            pixel_size = max(4, int(ratio * 40))
            small = cv2.resize(result, (w // pixel_size, h // pixel_size), interpolation=cv2.INTER_LINEAR)
            result = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)

    return result
# -----------------------------------------------------------

In [ ]:
%%capture
### TODO

def filter_task(image, face_landmarks_list):
    if not face_landmarks_list: return image
    h, w, _ = image.shape
    result = image.copy()
    #for face_landmarks in face_landmarks_list:
      #..................

    return result


In [ ]:
# Execution
run_engine(filter_example1)

<IPython.core.display.Javascript object>